In [24]:
import random

num = random.randint(1, 25)
print(num)


1


In [ ]:
# ===========================
# Part VIII: Python Practice – Linear Regression
# ===========================

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---------------------------
# Step 1: Dataset Creation
# ---------------------------
# Synthetic dataset with 1 feature for simplicity
np.random.seed(42)  # For reproducibility

n_samples = 100
X = 2 * np.random.rand(n_samples, 1)  # Feature between 0 and 2
true_slope = 3.5
true_intercept = 1.2
noise = np.random.randn(n_samples, 1)  # Gaussian noise

y = true_intercept + true_slope * X + noise  # Linear relation with noise

# Visualize the dataset
plt.scatter(X, y)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Synthetic Linear Dataset')
plt.show()

# ---------------------------
# Step 2: Train-Test Split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---------------------------
# Step 3: Data Preprocessing
# ---------------------------
# Optional: Standardize features for gradient descent
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Add intercept term (column of ones) for OLS analytical solution
X_train_ols = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]
X_test_ols = np.c_[np.ones((X_test_scaled.shape[0], 1)), X_test_scaled]

# ---------------------------
# Step 4: Linear Regression – Analytical OLS
# ---------------------------
# beta = (X^T X)^{-1} X^T y
beta_hat = np.linalg.inv(X_train_ols.T @ X_train_ols) @ (X_train_ols.T @ y_train)
print("OLS coefficients:", beta_hat.ravel())

# Predictions
y_pred_ols = X_test_ols @ beta_hat

# ---------------------------
# Step 5: Linear Regression – Gradient Descent
# ---------------------------
def gradient_descent(X, y, lr=0.1, n_iter=1000):
    n_samples, n_features = X.shape
    beta = np.zeros((n_features, 1))
    for i in range(n_iter):
        gradient = (2 / n_samples) * X.T @ (X @ beta - y)
        beta -= lr * gradient
    return beta

beta_gd = gradient_descent(X_train_ols, y_train, lr=0.1, n_iter=1000)
print("Gradient Descent coefficients:", beta_gd.ravel())

# Predictions
y_pred_gd = X_test_ols @ beta_gd

# ---------------------------
# Step 6: Evaluation
# ---------------------------
def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2_score(y_true, y_pred):
    ss_total = np.sum((y_true - np.mean(y_true))**2)
    ss_res = np.sum((y_true - y_pred)**2)
    return 1 - ss_res / ss_total

mse_ols = mean_squared_error(y_test, y_pred_ols)
r2_ols = r2_score(y_test, y_pred_ols)

mse_gd = mean_squared_error(y_test, y_pred_gd)
r2_gd = r2_score(y_test, y_pred_gd)

print(f"OLS MSE: {mse_ols:.3f}, R2: {r2_ols:.3f}")
print(f"Gradient Descent MSE: {mse_gd:.3f}, R2: {r2_gd:.3f}")

# ---------------------------
# Step 7: Residual Analysis
# ---------------------------
residuals_ols = y_test - y_pred_ols
plt.scatter(y_pred_ols, residuals_ols)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Predicted values")
plt.ylabel("Residuals")
plt.title("Residual Plot (OLS)")
plt.show()

# ---------------------------
# Step 8: Visualization of Fit
# ---------------------------
plt.scatter(X_test_scaled, y_test, label='True Data')
plt.plot(X_test_scaled, y_pred_ols, color='r', label='OLS Prediction')
plt.plot(X_test_scaled, y_pred_gd, color='g', linestyle='--', label='GD Prediction')
plt.xlabel('X (scaled)')
plt.ylabel('y')
plt.title('Linear Regression Fit')
plt.legend()
plt.show()
